# Celltypes in spatial context

In [101]:
# Load Packages for data display
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from pathlib import Path

import json
import tifffile
import zarr
from matplotlib.colors import to_rgb

import os

In [54]:
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 100)

### Helper functions

In [ ]:
def bundle_for(sample):
    hits = [p for p in XENIUM_ROOT.glob(f"*/{sample}") if (p / "morphology_focus").is_dir()]
    if not hits:
        raise FileNotFoundError(f"no Xenium bundle with morphology_focus for {sample!r}")
    return sorted(hits, key=lambda p: p.parent.name != "run1")[-1]


def load_morphology(sample, channels=(0,), colors=None, gains=None,
                    level=3, crop=None, clip=(1.0, 99.5)):
    """Composite Xenium morphology channels into an RGB image.

    channels : ints 0-3, see CHANNEL_NAMES
    colors   : per-channel matplotlib color (default DEFAULT_COLORS)
    gains    : per-channel multiplier applied after contrast scaling (default 1.0)
    level    : pyramid level; 0 = full res (23860x19957), 3 ~ 8x down
    crop     : (x_start, y_start, x_end, y_end) in MICRONS, same frame as obsm["X_spatial"]
    clip     : (low, high) percentiles for per-channel contrast, computed on the crop

    Returns (rgb, extent): float32 HxWx3 in [0,1], and extent in microns for
    imshow(..., extent=extent, origin="upper").
    """
    channels = list(channels)
    colors = colors or [DEFAULT_COLORS[c] for c in channels]
    gains  = gains if gains is not None else [1.0] * len(channels)
    if not (len(colors) == len(gains) == len(channels)):
        raise ValueError("channels, colors and gains must be the same length")

    bundle = bundle_for(sample)
    px = json.load(open(bundle / "experiment.xenium"))["pixel_size"]

    rgb = extent = None
    for ch, color, gain in zip(channels, colors, gains):
        tif = bundle / "morphology_focus" / f"morphology_focus_{ch:04d}.ome.tif"
        if not tif.exists():
            raise FileNotFoundError(tif)

        arr = zarr.open(tifffile.imread(tif, is_ome=False, aszarr=True), mode="r")
        full_h, full_w = arr["0"].shape
        plane = arr[str(level)]
        lvl_h, lvl_w = plane.shape

        # microns per pixel at this level, from real shapes: pyramid steps are NOT exactly 2x
        mx, my = px * full_w / lvl_w, px * full_h / lvl_h

        if crop is None:
            r0, c0, r1, c1 = 0, 0, lvl_h, lvl_w
        else:
            x0, y0, x1, y1 = crop
            c0, c1 = sorted((int(np.floor(x0 / mx)), int(np.ceil(x1 / mx))))
            r0, r1 = sorted((int(np.floor(y0 / my)), int(np.ceil(y1 / my))))
            c0, c1 = max(c0, 0), min(c1, lvl_w)
            r0, r1 = max(r0, 0), min(r1, lvl_h)
            if c1 <= c0 or r1 <= r0:
                raise ValueError(
                    f"crop {crop} falls outside the image "
                    f"(0-{lvl_w*mx:.0f}, 0-{lvl_h*my:.0f} um)")

        img = np.asarray(plane[r0:r1, c0:c1]).astype(np.float32)   # lazy: only these tiles are read

        lo, hi = np.percentile(img, clip)
        img = np.zeros_like(img) if hi <= lo else np.clip((img - lo) / (hi - lo), 0, 1)
        img *= gain

        if rgb is None:
            rgb = np.zeros((*img.shape, 3), dtype=np.float32)
            extent = [c0 * mx, c1 * mx, r1 * my, r0 * my]
        rgb += img[..., None] * np.asarray(to_rgb(color), dtype=np.float32)

    return np.clip(rgb, 0, 1), extent


def cells_in_crop(adata, sample, crop=None, batch_key="batch", basis="X_spatial"):
    """Subset adata to one sample and, if given, to the crop box (microns)."""
    sub = adata[adata.obs[batch_key] == sample]
    if crop is None:
        return sub
    x0, y0, x1, y1 = crop
    xy = sub.obsm[basis]
    m = ((xy[:, 0] >= min(x0, x1)) & (xy[:, 0] <= max(x0, x1))
         & (xy[:, 1] >= min(y0, y1)) & (xy[:, 1] <= max(y0, y1)))
    return sub[m]


## Load the data

In [ ]:
os.getcwd()

In [ ]:
os.chdir('SET TO WORKING DICTIONARY')

In [5]:
spatial_ad_5k = sc.read_h5ad('data/xenium_controls_only.h5ad')

In [7]:
spatial_ad_5k

In [9]:
spatial_ad_5k.obs['batch'].value_counts()

In [11]:
# Load the hgca_gene_panel 
unique_si_genes = [
    'ABCA8', 'ACKR1', 'ACTA2', 'ADAMDEC1', 'ADGRG6', 'AICDA', 'ALDOB', 'ANO1', 
    'ANPEP', 'ANXA13', 'APOA1', 'APOB', 'APOC3', 'AREG', 'ASCL2', 'ATOH1', 
    'BANK1', 'BATF', 'BCL6', 'BCL7A', 'BEST4', 'BMP4', 'C1QA', 'C1QB', 'C1QC', 
    'C1orf54', 'C7', 'CA2', 'CA7', 'CA8', 'CAV1', 'CCK', 'CCL13', 'CCL19', 
    'CCL2', 'CCL20', 'CCL21', 'CCL22', 'CCL23', 'CCL25', 'CCL3', 'CCL8', 
    'CCNA2', 'CCR7', 'CD14', 'CD160', 'CD163L1', 'CD1C', 'CD207', 'CD209', 
    'CD24', 'CD244', 'CD27', 'CD300E', 'CD34', 'CD36', 'CD3D', 'CD3E', 'CD4', 
    'CD40', 'CD40LG', 'CD69', 'CD7', 'CD79A', 'CD83', 'CD8A', 'CD8B', 'CDH5', 
    'CDHR5', 'CFTR', 'CHGA', 'CHGB', 'CLC', 'CLCA1', 'CLDN5', 'CLEC10A', 
    'CLEC4C', 'CLEC9A', 'CLIC3', 'CLU', 'COL3A1', 'CPA3', 'CPE', 'CR1', 'CR2', 
    'CSF1', 'CSF2', 'CTLA4', 'CTSG', 'CX3CR1', 'CXCL13', 'CXCL14', 'CXCL2', 
    'CXCL5', 'CXCL6', 'CXCR4', 'CXCR5', 'DEFA5', 'DEFA6', 'DNASE1L3', 'DPT', 
    'DUOX2', 'EDN1', 'ELF3', 'EOMES', 'EPCAM', 'ERO1B', 'ETV1', 'F3', 'FABP1', 
    'FABP4', 'FBLN1', 'FCER1A', 'FCER1G', 'FCGR2B', 'FCN1', 'FCN3', 'FCRL4', 
    'FGFBP2', 'FLT1', 'FOLR2', 'FOXF2', 'FOXP3', 'FSCN1', 'FXYD3', 'G0S2', 
    'GAST', 'GATA3', 'GCG', 'GIP', 'GLI1', 'GNLY', 'GP2', 'GP9', 'GPR183', 
    'GREM1', 'GUCA2A', 'GULP1', 'GZMA', 'GZMB', 'GZMH', 'GZMK', 'HAVCR2', 
    'HBA1', 'HBB', 'HCAR3', 'HELLS', 'HEPACAM2', 'HIGD1B', 'HLA-DRA', 'HMGB2', 
    'HOPX', 'HPGDS', 'ICAM1', 'ICOS', 'IDO1', 'IFIT2', 'IFNG', 'IGFBP3', 
    'IGFBP4', 'IGFBP7', 'IGHA1', 'IGHA2', 'IGHD', 'IGHE', 'IGHG1', 'IGHM', 
    'IGKC', 'IGLC2', 'IL10', 'IL13', 'IL15', 'IL17A', 'IL1B', 'IL2', 'IL21', 
    'IL22', 'IL23R', 'IL2RA', 'IL3RA', 'IL4', 'IL7R', 'INSL5', 'ITGA1', 'ITGAE', 
    'ITGAX', 'ITLN1', 'ITLN2', 'JCHAIN', 'KCNN3', 'KIR2DL4', 'KIT', 'KLF2', 
    'KLF4', 'KLK1', 'KLK15', 'KLRC2', 'KLRF1', 'KLRG1', 'KRT1', 'KRT19', 
    'KRT86', 'LAG3', 'LAMP3', 'LAYN', 'LCN2', 'LEF1', 'LEFTY1', 'LGALS3', 
    'LGR5', 'LILRA4', 'LITAF', 'LRMP', 'LUM', 'LYST', 'LYVE1', 'LYZ', 'MADCAM1', 
    'MCM5', 'MGP', 'MIA', 'MKI67', 'MLN', 'MMRN1', 'MORN5', 'MRC1', 'MS4A1', 
    'MS4A6A', 'MSLN', 'MSR1', 'MUC2', 'MUC6', 'MYH11', 'NCR1', 'NCR2', 'NCR3', 
    'NDUFA4L2', 'NEIL1', 'NEUROD1', 'NEUROG3', 'NKG7', 'NOS2', 'NOTCH1', 
    'NOTCH3', 'NPR2', 'NPY', 'NRG1', 'NTS', 'OLFM4', 'PARD6B', 'PBRM1', 
    'PCLAF', 'PCNA', 'PCSK1N', 'PDCD1', 'PDGFRA', 'PDGFRB', 'PECAM1', 'PGC', 
    'PHLDA2', 'PLA2G2A', 'PLAUR', 'PLCB4', 'PLD4', 'PLN', 'PLVAP', 'POSTN', 
    'POU2F3', 'PRAP1', 'PRF1', 'PROK2', 'PROX1', 'PRSS2', 'PTGDR2', 'PTGS2', 
    'PYY', 'RAMP2', 'RANK', 'REG3A', 'REG3G', 'REG4', 'RGCC', 'RGMB', 'RGS13', 
    'RGS5', 'RORA', 'RORC', 'RSPO3', 'S1PR5', 'SCG5', 'SCT', 'SELENOP', 'SELL', 
    'SELP', 'SEMA3A', 'SEMA3G', 'SH2D6', 'SHTN1', 'SIRPA', 'SLC26A2', 'SLC4A4', 
    'SMOC2', 'SOX4', 'SOX6', 'SOX8', 'SPI1', 'SPIB', 'SPINK4', 'SST', 'STMN1', 
    'TAGLN', 'TBX21', 'TCTN3', 'TFF1', 'TFF2', 'TFF3', 'TGM2', 'THY1', 'TIGIT', 
    'TM4SF1', 'TNF', 'TNFAIP2', 'TNFRSF13C', 'TNFSF13B', 'TNNC1', 'TOP2A', 
    'TPH1', 'TPSAB1', 'TPSB2', 'TRAJ33', 'TRAV1-2', 'TRAV24', 'TRDC', 'TRGC1', 
    'TRPA1', 'TRPM5', 'TUBA1A', 'VCAN', 'VSTM2A', 'VWF', 'WNT2', 'WNT5A', 
    'WNT5B', 'WT1', 'XBP1', 'XCR1', 'ZBTB16'
]
unique_li_genes = [
    'ABCA8', 'ACKR4', 'ACTG2', 'ADAMDEC1', 'ADGRG6', 'ADH1', 'AGC', 'AGR2', 
    'AICDA', 'ALDH1', 'ANO1', 'ANXA13', 'APOD', 'AQP8', 'AREG', 'ASCL2', 
    'ATP4', 'BATF', 'BCAN', 'BCL6', 'BCL7', 'BEST4', 'BMP4', 'C7', 'CA1', 
    'CA2', 'CA7', 'CA8', 'CAV1', 'CBLIF', 'CCL20', 'CCL21', 'CCL22', 'CCL23', 
    'CCL25', 'CCL3', 'CCNA2', 'CCR7', 'CD1', 'CD160', 'CD163', 'CD19', 'CD207', 
    'CD209', 'CD24', 'CD244', 'CD27', 'CD3', 'CD300', 'CD34', 'CD4', 'CD40', 
    'CD69', 'CD7', 'CD79', 'CD8', 'CD83', 'CDH19', 'CDH5', 'CEACAM7', 'CHGA', 
    'CHGB', 'CLC', 'CLCA1', 'CLDN3', 'CLDN5', 'CLEC10', 'CLEC4', 'CLEC9', 
    'CLIC3', 'CLL2', 'CLL3', 'CLU', 'COL3', 'COLEC12', 'CPA3', 'CPE', 'CR1', 
    'CR2', 'CRTH2', 'CRYAB', 'CSF2', 'CSPG4', 'CTLA4', 'CTSG', 'CX3', 'CXCL13', 
    'CXCL14', 'CXCL2', 'CXCL5', 'CXCL6', 'CXCR4', 'CXCR5', 'DES', 'DNASE1', 
    'EDN1', 'ENO2', 'EOMES', 'ETV1', 'F3', 'FABP1', 'FABP4', 'FBLN1', 'FCER1', 
    'FCGR2', 'FCGR3', 'FCN1', 'FCN3', 'FLT1', 'FOLR2', 'FOXP3', 'FRZB', 
    'FSCN1', 'FUT9', 'FXYD1', 'FXYD3', 'GATA3', 'GCG', 'GLI1', 'GP2', 
    'GPR155', 'GPR183', 'GREM1', 'GUCA2', 'GULP1', 'GZMA', 'GZMB', 'GZMH', 
    'GZMK', 'HAVCR2', 'HBB', 'HCAR3', 'HELLS', 'HEPACAM2', 'HMGB2', 'HOPX', 
    'HPGDS', 'ICAM1', 'ICAM2', 'ICOS', 'IDO1', 'IFIT2', 'IFNG', 'IGFBP3', 
    'IGFBP4', 'IGFBP7', 'IGHD', 'IGKC', 'IGLC2', 'IKZF2', 'IL1', 'IL10', 
    'IL13', 'IL15', 'IL17', 'IL1R1', 'IL21', 'IL22', 'IL23', 'IL2RA', 'IL3', 
    'IL4', 'IL7', 'INSL5', 'IRAG2', 'ITGA1', 'ITGA2', 'ITGAE', 'ITGAX', 
    'ITLN1', 'KCNN3', 'KCNS3', 'KIR2', 'KIT', 'KLF2', 'KLF4', 'KLK1', 'KLRC2', 
    'KLRD1', 'KLRF1', 'KLRG1', 'KLRK1', 'KRT1', 'KRT86', 'LAMP3', 'LAYN', 
    'LCN2', 'LEF1', 'LEFTY1', 'LGR5', 'LIPF', 'LITAF', 'LRMP', 'LUM', 'LYST', 
    'LYVE1', 'LYZ', 'MADCAM1', 'MBP', 'MCAM', 'MCM5', 'MGP', 'MIA', 'MKI67', 
    'MMP9', 'MMRN1', 'MORN5', 'MRC1', 'MS4', 'MSR1', 'MUC2', 'MUC5', 'MUC6', 
    'MUSTN1', 'MYH11', 'NCR1', 'NCR2', 'NCR3', 'NEIL1', 'NEUROD1', 'NKG7', 
    'NOS2', 'NOTCH1', 'NOTCH3', 'NPR2', 'NPY', 'NRG1', 'NT5', 'NTRK2', 'OLFM4', 
    'OTOP2', 'PCLAF', 'PCNA', 'PCSK1', 'PDCD1', 'PDGFRA', 'PDGFRB', 'PECAM1', 
    'PGA3', 'PHLDA2', 'PLA2', 'PLAC8', 'PLAUR', 'PLCB4', 'PLD4', 'PLP1', 
    'POSTN', 'PRBM1', 'PRF1', 'PROX1', 'PTGDR2', 'PTGDS', 'PTGS2', 'PYY', 
    'RAMP2', 'RANK', 'RBFOX3', 'REG3', 'REG4', 'RERGL', 'RGCC', 'RGMB', 
    'RGS13', 'RLBP1', 'RORC', 'S100', 'SCG5', 'SELENOP', 'SELL', 'SELP', 
    'SEMA3', 'SFRP5', 'SHTN1', 'SIRPA', 'SLC12', 'SLC17', 'SLC26', 'SLC4', 
    'SMOC2', 'SNAP25', 'SOX10', 'SOX4', 'SOX6', 'SOX8', 'SPI1', 'SPIB', 
    'SPINK4', 'SPINK5', 'SPP1', 'STAB2', 'STMN', 'STMN1', 'TAGLN', 'TBX21', 
    'TCTN3', 'TFF1', 'TFF2', 'TFF3', 'TGFB', 'TGM2', 'THY1', 'TIGIT', 'TNF', 
    'TNFAIP2', 'TNFRSF13', 'TNFSF11', 'TNFSF13', 'TNNC1', 'TOP2', 'TPH1', 
    'TPSAB1', 'TPSB2', 'TRAJ33', 'TRAV1-2', 'TRAV24', 'TRDC', 'TRDV1', 
    'TRDV2', 'TRDV3', 'TRPA1', 'TRPM5', 'TSPAN8', 'TUBA1', 'VSTM2', 'VWF', 
    'WNT2', 'WNT5', 'XBP1', 'XCR1', 'ZBTB16', 'TF'
]


In [60]:
# Load cell type reference

cell_type_ref_df = pd.read_csv('data/hgca_cell_type_reference.tsv', sep='\t')

In [108]:
cell_type_ref_df.head()

In [90]:
# build the final cell type marker ref dictionary

cell_type_marker_dict ={}

for ct in cell_type_ref_df['hgca_celltype_v1']:
    de_genes=list(cell_type_ref_df.loc[cell_type_ref_df['hgca_celltype_v1']==ct, 'hgca_celltype_v1--marker_gene_evidence'])[0].split(',')
    canonical_genes=list(cell_type_ref_df.loc[cell_type_ref_df['hgca_celltype_v1']==ct, 'hgca_celltype_v1--canonical_marker_genes'])[0].split(',')
    genes_combined = set(de_genes+canonical_genes)
    cell_type_marker_dict[ct]=list(genes_combined)


In [91]:
cell_type_marker_dict

## Generate images fpr all cell types

In [102]:
XENIUM_ROOT = Path("SET TO PATH OF RAW XENIUM OUTPUT DIRECTORY")

CHANNEL_NAMES  = {0: "DAPI", 1: "ATP1A1/CD45/E-Cadherin", 2: "18S", 3: "alphaSMA/Vimentin"}
DEFAULT_COLORS = {0: "#bcbec4", 1: "#989b9c", 2: "#5B5E5E", 3: "#2a2c2b"}


In [103]:
crop_dict = {
    'output-sample1':[0, 3300, 3000, 5000],
    'output-sample2':[1300, 2800, 4300, 4600],
    'output-sample3':[300, 0, 3300, 1700],
    'output-sample4':[1500, 1850, 4500, 3550]
}


In [104]:
FIGDIR = Path("images/celltype_explorer/spatial/")          # pick your location
FIGDIR.mkdir(parents=True, exist_ok=True)

In [105]:
import matplotlib.patheffects as pe
from scipy.sparse import issparse
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.font_manager import FontProperties
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

In [106]:
MIN_CELLS    = 5
SCALEBAR_UM  = 500          # all four crops are 3000 um wide -> 1/6 of panel width

SAMPLES = [
    'output-sample1',
    'output-sample2',
    'output-sample3',
    'output-sample4',
]

# ============================================================================
# ONE-TIME CACHE — morphology + cell coords depend only on the sample, not on
# the gene, so they are built once instead of once per (celltype, gene).
# ============================================================================
panel_cache = []
for s in SAMPLES:
    sub = cells_in_crop(spatial_ad_5k, s, crop=crop_dict[s])
    rgb, ext = load_morphology(s, channels=(0, 1, 2, 3),
                               gains=(0.5, 0.5, 0.5, 0.65), level=0, crop=crop_dict[s])
    rgb = (rgb * 255).astype(np.uint8)     # <- optional: 4x less RAM, imshow-safe

    X = sub.X
    X = X.tocsc() if issparse(X) else np.asarray(X)   # CSC -> cheap column slicing per gene

    panel_cache.append(dict(s=s, rgb=rgb, ext=ext,
                            xy=np.asarray(sub.obsm["X_spatial"]),
                            X=X, n_total=int(sub.n_obs)))
    print(f"cached {s}: {sub.n_obs} cells, image {rgb.shape[1]}x{rgb.shape[0]} px")

gene_idx = {g: i for i, g in enumerate(spatial_ad_5k.var_names)}
var_set  = set(spatial_ad_5k.var_names)


In [107]:


def expr_of(p, gene):
    """Expression vector for one gene in one cached panel."""
    col = p["X"][:, gene_idx[gene]]
    return np.asarray(col.todense()).ravel() if issparse(col) else np.asarray(col).ravel()


def add_scalebar(ax, um=SCALEBAR_UM, fontsize=14):
    """Horizontal scale bar. Length is in data units (= microns, since `extent`
    comes from experiment.xenium pixel_size); placement is anchored in axes
    fraction, which survives the datalim expansion caused by
    set_aspect('equal') + set_box_aspect()."""
    sb = AnchoredSizeBar(ax.transData, um, f"{um} \u00b5m", loc="lower right",
                         color="white", frameon=False, size_vertical=12,
                         pad=0.4, borderpad=0.6, sep=4,
                         fontproperties=FontProperties(size=fontsize))
    stroke = [pe.withStroke(linewidth=2.5, foreground="black")]
    sb.txt_label.get_children()[0].set_path_effects(stroke)
    for artist in sb.size_bar.get_children():
        artist.set_path_effects(stroke)
    ax.add_artist(sb)
    return sb



In [ ]:

# ============================================================================
# PLOTTING — only the expression values are recomputed per gene.
# ============================================================================

for ct in list(cell_type_marker_dict.keys()):
    print('Checking markers for', ct)

    for gene in cell_type_marker_dict[ct]:
        if gene not in var_set:
            continue

        # --- per-gene: expression only, images reused from cache ---
        panels = []
        for p in panel_cache:
            expr = expr_of(p, gene)
            pos  = expr > 0
            panels.append({**p, "expr": expr, "pos": pos, "n": int(pos.sum())})

        # --- gene-level guard: plot as soon as ONE sample has >= MIN_CELLS positive cells ---
        qualifying = [p for p in panels if p["n"] >= MIN_CELLS]
        if not qualifying:
            print(f"  {gene}: no sample with >= {MIN_CELLS} non-zero cells — skipping gene")
            continue

        # --- shared colour scale from the QUALIFYING samples only ---
        pooled = np.concatenate([p["expr"][p["pos"]] for p in qualifying])
        vmin, vmax = float(pooled.min()), float(np.percentile(pooled, 98))
        if vmax <= vmin:
            vmax = vmin + 1
        norm = Normalize(vmin=vmin, vmax=vmax)

        # --- one figure per gene: 1x4 grid, identical panel sizes, shared scale + colorbar ---
        fig, axes = plt.subplots(1, 4, figsize=(13, 4), constrained_layout=True)
        mappable = None
        for ax, p in zip(axes.ravel(), panels):
            ax.imshow(p["rgb"], extent=p["ext"], origin="upper")                 # background always
            pct = 100 * p["n"] / p["n_total"] if p["n_total"] else 0.0
            if p["n"] >= MIN_CELLS:
                mappable = ax.scatter(p["xy"][p["pos"], 0], p["xy"][p["pos"], 1],
                                      c=p["expr"][p["pos"]], s=5.5, cmap="magma",
                                      norm=norm, alpha=1, linewidths=0)          # same norm -> shared scale
                tag = f"n={p['n']} ({pct:.1f}%)"
            else:
                tag = f"n={p['n']} ({pct:.1f}%, <{MIN_CELLS} cells)"
            ax.set_aspect("equal")        # no tissue distortion
            ax.set_box_aspect(3/4)
            ax.set_facecolor("black")     # letterbox margins blend with morphology
            ax.set_title(f"{tag}", fontsize=18)
            ax.set_xticks([]); ax.set_yticks([])

        add_scalebar(axes[-1])            # rightmost panel only; all crops are 3000 um wide

        if mappable is None:              # safety fallback (qualifying is non-empty, so won't trigger)
            mappable = ScalarMappable(norm=norm, cmap="magma")
        cbar = fig.colorbar(mappable, ax=axes, orientation="horizontal",
                            location="bottom", shrink=0.35, pad=0.02, aspect=40)
        cbar.set_label("Expression per cell", fontsize=18)
        cbar.ax.tick_params(labelsize=16)
        fig.suptitle(f"{ct} — {gene}", fontsize=20)

        # stem = f"{ct}__{gene}".replace(" ", "_").replace("/", "-").lower()
        # fig.savefig(FIGDIR / f"{stem}.svg", dpi=150, bbox_inches="tight")
        stem = f"{ct}__{gene}".replace(" ", "_").replace("/", "-").lower()
        fig.savefig(FIGDIR / f"{stem}.webp", dpi=150, bbox_inches="tight",
                    facecolor="white",
                    pil_kwargs={"quality": 80, "method": 6})

        plt.show()
        plt.close(fig)
